# Phase 3: Behavioral Clustering

Train the fixed four-cluster K-Means pipeline, inspect scaled centroids, and compare average gold trajectories. Cluster names remain undecided until the outputs are reviewed.

In [1]:
from pathlib import Path
import sys

import duckdb
import joblib
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = 'notebook_connected'

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models import DB_PATH, FEATURE_COLS, MODELS_DIR, train_and_persist

## Train and persist

The notebook calls the production training function so model parameters, validation, artifacts, and DuckDB labels have one implementation.

In [2]:
with duckdb.connect(str(DB_PATH)) as conn:
    raw_profile, silhouette = train_and_persist(conn)

raw_profile.round(3)

Cluster sizes     : {0: 166, 1: 103, 2: 7, 3: 52}
Silhouette score  : 0.245
Feature means by cluster:
            gold_delta  total_deaths  deaths_while_ahead  tilt_spiral_ratio  max_death_streak  total_roams  avg_cs_sacrifice  roam_impact_rate  tilt_index
cluster_id                                                                                                                                                
0             -438.429         9.777               0.114              0.669             4.783        0.175             0.000             0.461       0.539
1              241.196         4.485               0.466              0.244             1.010        0.126             0.005             0.466       0.472
2              166.379         9.143               0.857              0.621             4.000        1.000             1.585             0.714       0.543
3              899.450         8.500               2.423              0.610             3.596        0.385             0.00

,gold_delta,total_deaths,deaths_while_ahead,tilt_spiral_ratio,max_death_streak,total_roams,avg_cs_sacrifice,roam_impact_rate,tilt_index
cluster_id,,,,,,,,,
0,-438.429,9.777,0.114,0.669,4.783,0.175,0.000,0.461,0.539
1,241.196,4.485,0.466,0.244,1.010,0.126,0.005,0.466,0.472
2,166.379,9.143,0.857,0.621,4.000,1.000,1.585,0.714,0.543
3,899.450,8.500,2.423,0.610,3.596,0.385,0.000,0.654,0.602


## Scaled centroid heatmap

K-Means is fitted in standardized feature space, so these centroids show which behaviors are above or below the dataset mean within each cluster.

In [3]:
model = joblib.load(MODELS_DIR / 'kmeans.pkl')
scaled_centroids = pd.DataFrame(
    model.cluster_centers_,
    columns=FEATURE_COLS,
    index=pd.Index(range(model.n_clusters), name='cluster_id'),
)

centroid_figure = px.imshow(
    scaled_centroids.T,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0,
    labels={'x': 'Cluster', 'y': 'Feature', 'color': 'Scaled mean'},
    title='Scaled K-Means centroids',
)
centroid_figure.show()

## Cluster summary

In [4]:
with duckdb.connect(str(DB_PATH), read_only=True) as conn:
    cluster_summary = conn.execute(
        '''
        SELECT
            cl.cluster_id,
            COUNT(*) AS games,
            AVG(CASE WHEN fm.win THEN 1.0 ELSE 0.0 END) AS win_rate,
            AVG(fm.gold_delta) AS avg_gold_delta,
            AVG(fm.total_deaths) AS avg_deaths
        FROM cluster_labels cl
        JOIN feature_matrix fm ON fm.match_id = cl.match_id
        GROUP BY cl.cluster_id
        ORDER BY cl.cluster_id
        '''
    ).df()

cluster_summary.round(3)

,cluster_id,games,win_rate,avg_gold_delta,avg_deaths
0,0,166,0.373,-438.429,9.777
1,1,103,0.777,241.196,4.485
2,2,7,0.429,166.379,9.143
3,3,52,0.519,899.450,8.500


## Average gold trajectories

In [5]:
with duckdb.connect(str(DB_PATH), read_only=True) as conn:
    gold_trajectories = conn.execute(
        '''
        SELECT
            cl.cluster_id,
            mt.timestamp_min,
            AVG(CAST(mt.gold AS DOUBLE)) AS avg_gold
        FROM cluster_labels cl
        JOIN match_timelines mt ON mt.match_id = cl.match_id
        GROUP BY cl.cluster_id, mt.timestamp_min
        ORDER BY cl.cluster_id, mt.timestamp_min
        '''
    ).df()

gold_trajectories['cluster_id'] = gold_trajectories['cluster_id'].astype(str)

trajectory_figure = px.line(
    gold_trajectories,
    x='timestamp_min',
    y='avg_gold',
    color='cluster_id',
    markers=False,
    labels={
        'timestamp_min': 'Game minute',
        'avg_gold': 'Average total gold',
        'cluster_id': 'Cluster',
    },
    title='Average gold trajectory by cluster',
)
trajectory_figure.show()

## Interpretation gate

Before naming clusters, compare centroid direction, cluster size, win rate, and trajectory shape. A useful name must describe observed behavior rather than imply causation.